# 1ピクセルでの保存モーメント経路の比較

既存のLiteBIRD標準スキャンによる全天畳み込みと同じ入力・同じ条件で、指定した1ピクセルだけを比較します。固定版全天mapとの比較なので、昨日のfixed計算と同じ `8b8d460` の環境を使用します。

- 通常経路: `convolve_ring_gpu` がscan momentsを内部生成
- 保存経路: scan momentsをホストへコピーしてJLSへ保存し、再読込したものから同じ畳み込みを再構成
- 最後に既存の `stokes_map.jls` の同じピクセルとも比較

`Float32`, `P=8`, `lmax=1200`, `nmax=120`, `κ=20`, 19 Hz, 1年、5 sky、5 beam、`rcond=1e-10` を既存結果に合わせています。

In [ ]:
using Pkg
using Serialization
using TOML
using Dates
using LinearAlgebra
using Statistics

function find_repo_root(start=pwd())
    current = abspath(start)
    while true
        if isfile(joinpath(current, "Project.toml")) &&
            isfile(joinpath(current, "src", "EBCoDeCMB.jl"))
            return current
        end
        parent = dirname(current)
        parent == current && error("EBCoDeCMB.jl repository root was not found")
        current = parent
    end
end

REPO = find_repo_root()
PERF_ROOT = joinpath(dirname(REPO), "test", "performance_test")
Pkg.activate(joinpath(PERF_ROOT, "falcons_ebcode_env-main-8b8d460"); io=devnull)

using CUDA
using EBCoDeCMB
using Falcons
using Healpix
using NPZ

include(joinpath(PERF_ROOT, "run_litebird_realistic_year_benchmark.jl"))
CUDA.functional() || error("A functional CUDA GPU is required")
println("repository = ", REPO)
println("CUDA device = ", CUDA.device())

## 1. 既存全天結果と同じ条件を設定

In [ ]:
nside = 512
lmax = 1200
bnmax = 120
kappa = 20
beta_order = 8
beta_max = 0.0035
sampling_rate_hz = 19.0
duration_seconds = 365 * 24 * 60 * 60
rcond = 1e-10
target_pixel = 1572617
scan_chunk_samples = 1_641_600
l_tile = 128

cp = ConvolutionParams(
    nside_output=nside,
    lstart=0,
    lstop=lmax,
    lmax=lmax,
    numofsky=5,
    numofbeam=5,
    bnmax=bnmax,
    κ=kappa,
    HWP=false,
)
scan_plan = ScanMomentPlan(
    kappa=cp.κ,
    nmax=calc_nmax(cp, cp.lstop),
    beta_order=beta_order,
    beta_max=beta_max,
    precision=Float32,
)
sample_count = round(Int, sampling_rate_hz * duration_seconds)
@assert sample_count == 599_184_000

sky_directory = joinpath(PERF_ROOT, "inputs", "CMB", "lmax1200")
beam_directory = joinpath(PERF_ROOT, "inputs", "beams", "elliptical_fwhm0p2deg_lmax1200")
sky_paths = [joinpath(sky_directory, "alm=CMB=r0=nside512=seed$(seed).npy") for seed in 0:4]
q_values = (1.000, 0.975, 0.950, 0.925, 0.900)
beam_paths = [joinpath(beam_directory, "beam$(i - 1)_q$( @sprintf("%.3f", q) ).npy") for (i, q) in enumerate(q_values)]
all(isfile, (sky_paths..., beam_paths...)) || error("one or more realistic input files are missing")

existing_map_path = joinpath(
    PERF_ROOT,
    "results",
    "realistic_cmb_elliptical_falcons_litebird_one_year_19hz_kappa20_float32_rcond1e-10_fixed_main_8b8d460_2026-09-09_retry1",
    "stokes_map.jls",
)
isfile(existing_map_path) || error("existing all-sky map was not found: $existing_map_path")

comparison_directory = joinpath(PERF_ROOT, "results", "one_pixel_persistent_scan_moments_comparison")
mkpath(comparison_directory)
moment_path = joinpath(comparison_directory, "pixel$(target_pixel)_scan_moments.jls")
summary_path = joinpath(comparison_directory, "comparison.toml")

(cp=cp, scan_plan=scan_plan, sample_count=sample_count, target_pixel=target_pixel, existing_map_path=existing_map_path)

## 2. Sky、beam、Wigner tableを準備

In [ ]:
function load_teb(path)
    raw = NPZ.npzread(path)
    expected = (lmax + 1) * (lmax + 2) ÷ 2
    size(raw) == (3, expected) || error("unexpected shape for $path: $(size(raw))")
    return Matrix{ComplexF32}(raw)
end

sky_teb = [load_teb(path) for path in sky_paths]
beam_teb = [load_teb(path) for path in beam_paths]
prepared_alm = prepare_alm(cp, sky_teb; input_basis=:teb, precision=Float32)
prepared_blm = prepare_blm(cp, beam_teb; input_basis=:teb, precision=Float32)
alm_gpu = upload_alm_gpu(prepared_alm)
wigner_table_host = build_wigner_chebyshev_table_cpu(cp, scan_plan)
wigner_table = upload_wigner_table_gpu(wigner_table_host)
println("prepared input: ", prepared_alm, " / ", prepared_blm)
println("Wigner table built")

## 3. 1年分のFalcons pointingsから対象ピクセルだけを抽出

pointing生成は既存全天ベンチマークと同じですが、ここでは全サンプルを保持せず、対象ピクセルの `φ, θ, ψ` だけを保存します。

In [ ]:
function pixel_ring(cp, pixel)
    1 <= pixel <= cp.resol.numOfPixels || error("pixel is outside the map")
    lo, hi = 1, ring_count(cp)
    while lo <= hi
        mid = (lo + hi) ÷ 2
        first_pixel, last_pixel = ring_pixel_range(cp, mid)
        if pixel < first_pixel
            hi = mid - 1
        elseif pixel > last_pixel
            lo = mid + 1
        else
            return mid
        end
    end
    error("could not find ring for pixel $pixel")
end

function collect_target_pointings(scan, cp, target_pixel; chunk_samples)
    phi_target = Float64[]
    theta_target = Float64[]
    psi_target = Float64[]
    resolution = cp.resol
    processed = 0
    next_report = 60_000_000
    for (phi, theta, psi) in falcons_litebird_scan_chunks(
        scan,
        sample_count;
        chunk_samples=chunk_samples,
    )
        @inbounds for sample_index in eachindex(phi, theta, psi)
            pixel = Healpix.ang2pixRing(
                resolution,
                Float64(theta[sample_index]),
                Float64(phi[sample_index]),
            )
            if pixel == target_pixel
                push!(phi_target, Float64(phi[sample_index]))
                push!(theta_target, Float64(theta[sample_index]))
                push!(psi_target, Float64(psi[sample_index]))
            end
        end
        processed += length(phi)
        if processed >= next_report
            println("pointings: $processed / $sample_count, target hits: $(length(phi_target))")
            next_report += 60_000_000
            flush(stdout)
        end
    end
    return phi_target, theta_target, psi_target, processed
end

scan = litebird_falcons_strategy(
    sampling_rate_hz=sampling_rate_hz,
    duration_seconds=duration_seconds,
    nside=nside,
)
phi_target, theta_target, psi_target, processed = collect_target_pointings(
    scan,
    cp,
    target_pixel;
    chunk_samples=scan_chunk_samples,
)
processed == sample_count || error("not all scan samples were processed")
isempty(phi_target) && error("target pixel has no hits")
target_angles = Angles(cp; pix=target_pixel, φ=phi_target, θ=theta_target, ψ=psi_target)
target_ring = pixel_ring(cp, target_pixel)
geometry = ring_geometry(cp, target_ring)
position_in_ring = target_pixel - geometry.first_pixel + 1
active = ActiveRingPixels([position_in_ring], geometry.pixel_count)
println("target pixel=$(target_pixel), ring=$(target_ring), hits=$(target_angles.num), beta_max_seen=$(maximum(target_angles.β))")

## 4. 通常経路の1ピクセル畳み込み

In [ ]:
direct_result = convolve_ring_gpu(
    cp,
    geometry,
    active,
    [target_angles],
    alm_gpu,
    prepared_blm,
    wigner_table,
    scan_plan;
    l_tile=l_tile,
    moment_method=:gemm,
)
direct_accumulated = Array(direct_result.accumulated)[:, :, 1, :]
direct_stokes = Array(ring_stokes(direct_result, [target_angles]; rcond=rcond, precision=Float32))[:, :, :, 1]
free_ring_convolution!(direct_result)
println("direct one-pixel convolution completed")
direct_accumulated

## 5. Scan momentsを保存して再読込

保存する配列の軸は `(scan_weight, beta_mode, q, n)`、ここでは `2 × P × (2κ+1) × (2nmax+1)` です。

In [ ]:
moments_gpu = scan_moments_gpu(scan_plan, [target_angles]; method=:gemm)
moments_host = Array(moments_gpu)
CUDA.unsafe_free!(moments_gpu)
saved_h2 = (1.0 / target_angles.num) * sum(cis(2 * psi) for psi in target_angles.ψ)
saved_h4 = (1.0 / target_angles.num) * sum(cis(4 * psi) for psi in target_angles.ψ)
moment_payload = (
    pixel=target_pixel,
    hits=target_angles.num,
    kappa=cp.κ,
    nmax=scan_plan.nmax,
    beta_order=scan_plan.beta_order,
    beta_max=Float64(scan_plan.beta_max),
    precision="Float32",
    h2=saved_h2,
    h4=saved_h4,
    moments=moments_host,
)
open(moment_path, "w") do io
    serialize(io, moment_payload)
end
reloaded_payload = open(deserialize, moment_path)
reloaded_moments = reloaded_payload.moments
@assert reloaded_moments == moments_host
@assert reloaded_payload.h2 == saved_h2
@assert reloaded_payload.h4 == saved_h4
println("saved moments: ", size(reloaded_moments), " / ", filesize(moment_path), " bytes")
size(reloaded_moments)

## 6. 保存済みmomentsから同じ1ピクセル畳み込みを再構成

ここでは `convolve_ring_gpu` の内部処理を、公開されている `build_effective_beam_gpu`、`build_sky_ring_gpu`、`select_active_sky_ring_gpu`、`accumulate_convolution_tile_gemm_gpu!` の順に明示的に呼び出します。したがって、違いはscan momentsの生成を再実行せず、保存配列をGPUへ戻して使う点だけです。

In [ ]:
function convolve_from_saved_moments(
    cp,
    geometry,
    active,
    angles,
    moments_host,
    alm_gpu,
    prepared_blm,
    wigner_table;
    l_tile,
)
    moments_gpu = CuArray(moments_host)
    ringd = ringd_blocks_recursion_gpu(cp, geometry.theta; precision=Float32)
    accumulated = CUDA.zeros(ComplexF32, cp.numofsky, cp.numofbeam, active.active_count, 2)
    succeeded = false
    try
        for l_first in cp.lstart:l_tile:cp.lstop
            l_last = min(l_first + l_tile - 1, cp.lstop)
            effective = nothing
            sky_ring = nothing
            active_sky_ring = nothing
            try
                effective = build_effective_beam_gpu(
                    wigner_table,
                    moments_gpu,
                    prepared_blm,
                    l_first,
                    l_last,
                )
                sky_ring = build_sky_ring_gpu(
                    cp,
                    ringd,
                    alm_gpu,
                    geometry.phi_pixels,
                    l_first,
                    l_last,
                )
                active_sky_ring = select_active_sky_ring_gpu(sky_ring, active)
                accumulate_convolution_tile_gemm_gpu!(
                    accumulated,
                    active_sky_ring,
                    effective,
                )
            finally
                if active_sky_ring !== nothing && sky_ring !== nothing
                    free_active_sky_ring!(active_sky_ring, sky_ring)
                end
                sky_ring === nothing || CUDA.unsafe_free!(sky_ring)
                effective === nothing || CUDA.unsafe_free!(effective)
            end
        end
        CUDA.synchronize()
        positions = Int.(Array(active.positions))
        pixels = geometry.first_pixel .+ positions .- 1
        result = RingConvolution{Float32}(
            geometry.ring,
            geometry.first_pixel,
            geometry.pixel_count,
            positions,
            pixels,
            accumulated,
        )
        succeeded = true
        return result
    finally
        CUDA.unsafe_free!(moments_gpu)
        CUDA.unsafe_free!(ringd)
        succeeded || CUDA.unsafe_free!(accumulated)
    end
end

function stokes_from_saved_psi(accumulated, h2, h4; rcond)
    h_matrix = ComplexF64[
        1.0              0.5*conj(h2)     0.5*h2
        0.5*h2           0.25             0.25*h4
        0.5*conj(h2)     0.25*conj(h4)    0.25
    ]
    condition = cond(h_matrix)
    (!isfinite(condition) || inv(condition) <= rcond) &&
        error("saved psi information is insufficient for map-making")
    factorization = lu(h_matrix)
    sky_count, beam_count = size(accumulated, 1), size(accumulated, 2)
    stokes = zeros(Float32, sky_count, beam_count, 3)
    for beam_index in 1:beam_count, sky_index in 1:sky_count
        spin0 = ComplexF64(accumulated[sky_index, beam_index, 1])
        half_spin2 = ComplexF64(accumulated[sky_index, beam_index, 2]) / 2
        solution = factorization \
            ComplexF64[spin0, half_spin2, conj(half_spin2)]
        stokes[sky_index, beam_index, 1] = Float32(real(solution[1]))
        stokes[sky_index, beam_index, 2] = Float32(real(solution[2]))
        stokes[sky_index, beam_index, 3] = Float32(imag(solution[2]))
    end
    return stokes
end

saved_result = convolve_from_saved_moments(
    cp,
    geometry,
    active,
    [target_angles],
    reloaded_moments,
    alm_gpu,
    prepared_blm,
    wigner_table;
    l_tile=l_tile,
)
saved_accumulated = Array(saved_result.accumulated)[:, :, 1, :]
saved_stokes = stokes_from_saved_psi(saved_accumulated, reloaded_payload.h2, reloaded_payload.h4; rcond=rcond)
free_ring_convolution!(saved_result)
println("saved-moment one-pixel convolution completed")

## 7. 通常経路・保存経路・既存全天mapを比較

In [ ]:
existing_map = open(deserialize, existing_map_path)
existing_pixel = existing_map[:, :, :, target_pixel]

moment_reload_max_abs = Float64(maximum(abs.(reloaded_moments .- moments_host)))
accumulated_max_abs = Float64(maximum(abs.(saved_accumulated .- direct_accumulated)))
accumulated_rms = sqrt(Float64(sum(abs2, saved_accumulated .- direct_accumulated)) / length(saved_accumulated))
stokes_saved_vs_direct_max_abs = Float64(maximum(abs.(saved_stokes .- direct_stokes)))
stokes_saved_vs_direct_rms = sqrt(Float64(sum(abs2, saved_stokes .- direct_stokes)) / length(saved_stokes))
stokes_saved_vs_fullsky_max_abs = Float64(maximum(abs.(saved_stokes .- existing_pixel)))
stokes_direct_vs_fullsky_max_abs = Float64(maximum(abs.(direct_stokes .- existing_pixel)))

angle_h2 = (1.0 / target_angles.num) * sum(cis(2 * psi) for psi in target_angles.ψ)
angle_h4 = (1.0 / target_angles.num) * sum(cis(4 * psi) for psi in target_angles.ψ)
h2 = reloaded_payload.h2
h4 = reloaded_payload.h4
saved_psi_h2_max_abs = Float64(abs(h2 - angle_h2))
saved_psi_h4_max_abs = Float64(abs(h4 - angle_h4))
moment_bytes = filesize(moment_path)
summary = Dict{String,Any}(
    "pixel" => target_pixel,
    "hits" => target_angles.num,
    "moment_shape" => collect(size(reloaded_moments)),
    "moment_file_bytes" => moment_bytes,
    "moment_reload_max_abs" => moment_reload_max_abs,
    "accumulated_saved_vs_direct_max_abs" => accumulated_max_abs,
    "accumulated_saved_vs_direct_rms" => accumulated_rms,
    "stokes_saved_vs_direct_max_abs" => stokes_saved_vs_direct_max_abs,
    "stokes_saved_vs_direct_rms" => stokes_saved_vs_direct_rms,
    "stokes_saved_vs_existing_fullsky_max_abs" => stokes_saved_vs_fullsky_max_abs,
    "stokes_direct_vs_existing_fullsky_max_abs" => stokes_direct_vs_fullsky_max_abs,
    "h2_real" => real(h2),
    "h2_imag" => imag(h2),
    "h4_real" => real(h4),
    "h4_imag" => imag(h4),
    "saved_psi_h2_max_abs" => saved_psi_h2_max_abs,
    "saved_psi_h4_max_abs" => saved_psi_h4_max_abs,
    "existing_map_path" => existing_map_path,
    "moment_path" => moment_path,
)
open(summary_path, "w") do io
    TOML.print(io, summary)
end
one_pixel_result_path = joinpath(comparison_directory, "pixel$(target_pixel)_stokes.jls")
open(one_pixel_result_path, "w") do io
    serialize(io, (pixel=target_pixel, direct=direct_stokes, saved=saved_stokes, existing_fullsky=existing_pixel))
end

println("saved moments vs direct accumulated max abs = ", accumulated_max_abs)
println("saved moments vs direct Stokes max abs = ", stokes_saved_vs_direct_max_abs)
println("saved moments vs existing full-sky Stokes max abs = ", stokes_saved_vs_fullsky_max_abs)
println("summary = ", summary_path)
println("one-pixel result = ", one_pixel_result_path)
summary

## 読み方

`saved_vs_direct` がFloat32丸め誤差レベルで小さければ、モーメントを保存して後から畳み込みへ渡す経路が、現行の通常経路と一致しています。 `h2` と `h4` はmap-makingに使う `ψ` の情報で、このnotebookでは別保存した `h2/h4` だけでStokes復元しています。実装を全天保存へ拡張する際は、少なくとも `h4`（およびhit count）をモーメントとは別に保存する必要があります。

既存全天mapとの差は、同じ `8b8d460` の規約を使っていることの確認です。全天側は複数GPU・自動l-tile、1ピクセル側は固定l-tileで計算しているため、Float32の還元順序による小さな差が残る可能性があります。保存方式そのものの判定には `saved_vs_direct` と、保存した `h2/h4` だけでのStokes復元を使います。